# as-strided-windowing — ex4: 2-D image patch grid via as_strided (im2col primitive)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-windowing`. Running the final beacon cell reports progress against the `PyTorch: as_strided windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: as_strided windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-windowing`** (exercise 4). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-windowing"
DD_SUBTOPIC = "PyTorch: as_strided windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## as_strided windowing — quick refresher

`x.as_strided(size, stride)` returns a view into `x.storage()` with *manually specified* shape and stride tuples — no bounds checking. For a 2-D image `(H, W)`, a `KxK` patch grid is `size=(H-K+1, W-K+1, K, K)` and `stride=(W, 1, W, 1)` — outer strides walk the patch origin, inner strides walk inside one patch. This is the canonical Conv-input im2col primitive.

### Exercise 4 — 2-D image patch grid via as_strided (im2col primitive)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `t.as_strided` along TWO spatial axes simultaneously to extract every KxK patch from an `(H, W)` image into an `(H-K+1, W-K+1, K, K)` view, the standard im2col primitive behind conv layers.
> Keywords: im2col, patch-extraction, 2d-windowing, conv-prep
> ```

**KCs targeted:** `as-strided-window-size`, `as-strided-window-stride`

ex1–ex3 covered 1-D sliding windows (compute size/stride, channelled extension, step>1). This one extends to a TRUE 2-D window pass — the im2col primitive behind `Conv2d`.

Implement `ex4_image_patches(img, K)`:

1. `img` has shape `(H, W)` — a single-channel image, contiguous.
2. Extract all `KxK` patches (stride 1 in both spatial dims) as a single view of shape `(H-K+1, W-K+1, K, K)`.
3. Use ONE `t.as_strided` call. The size tuple is `(H-K+1, W-K+1, K, K)`. The stride tuple — based on the fact that `img.stride()` is `(W, 1)` for a contiguous image — should be `(W, 1, W, 1)`:
   - outer two strides step the patch *origin* across the image one row / one column at a time
   - inner two strides step *within* a patch one row / one column at a time

4. Return the `(H-K+1, W-K+1, K, K)` view.

The test verifies one hand-picked patch, the global min/max, and a flatten-then-matmul use as a depthwise convolution sanity check.

In [ ]:
def ex4_image_patches(img: Tensor, K: int) -> Tensor:
    """Return (H-K+1, W-K+1, K, K) view of all KxK patches."""
    raise NotImplementedError()


def _test_ex4():
    img = t.arange(20, dtype=t.float32).reshape(4, 5)
    patches = ex4_image_patches(img, K=2)
    assert patches.shape == (3, 4, 2, 2), f'expected (3,4,2,2), got {tuple(patches.shape)}'
    assert patches.dtype == t.float32
    # Hand-checked patch [0, 0]: the top-left 2x2 corner.
    expected_00 = t.tensor([[0.0, 1.0], [5.0, 6.0]])
    assert t.allclose(patches[0, 0], expected_00), f'top-left patch wrong:\n{patches[0,0]}'
    # Hand-checked patch [2, 3]: the bottom-right 2x2 corner.
    expected_23 = t.tensor([[13.0, 14.0], [18.0, 19.0]])
    assert t.allclose(patches[2, 3], expected_23), f'bottom-right patch wrong:\n{patches[2,3]}'

    # Patch [1, 2]: starts at img[1, 2].
    expected_12 = t.tensor([[7.0, 8.0], [12.0, 13.0]])
    assert t.allclose(patches[1, 2], expected_12)

    # Larger K.
    img2 = t.arange(36, dtype=t.float32).reshape(6, 6)
    patches2 = ex4_image_patches(img2, K=3)
    assert patches2.shape == (4, 4, 3, 3)
    # Center-most patch at [1, 1] starts at img2[1, 1].
    assert t.allclose(patches2[1, 1, 0], img2[1, 1:4])
    assert t.allclose(patches2[1, 1, 2], img2[3, 1:4])

    # Storage-sharing sanity: it's a view, not a copy.
    img3 = t.arange(16, dtype=t.float32).reshape(4, 4).clone()
    p3 = ex4_image_patches(img3, K=2)
    img3[0, 0] = -999.0
    assert p3[0, 0, 0, 0].item() == -999.0, 'expected a view, got a copy'

    # Use as a depthwise sum-conv: sum over each patch should equal a hand-rolled box filter.
    rng = t.Generator().manual_seed(11)
    imrand = t.randn(8, 8, generator=rng)
    P = ex4_image_patches(imrand, K=3)
    sums = P.sum(dim=(-2, -1))
    # Compare to nested Python loop.
    expected_sums = t.empty(6, 6)
    for i in range(6):
        for j in range(6):
            expected_sums[i, j] = imrand[i:i+3, j:j+3].sum()
    assert t.allclose(sums, expected_sums, atol=1e-5), 'patch sums must match loop reference'

    # --- Visualization: box-blur the image via patch-mean ---
    H_v, W_v = 32, 32
    ys = t.linspace(-1, 1, H_v).unsqueeze(1).expand(H_v, W_v)
    xs = t.linspace(-1, 1, W_v).unsqueeze(0).expand(H_v, W_v)
    img_v = t.exp(-(xs ** 2 + ys ** 2) / 0.05) + 0.3 * t.randn(H_v, W_v, generator=rng)
    blurred = ex4_image_patches(img_v, K=5).mean(dim=(-2, -1))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))
    ax1.imshow(img_v.numpy(), cmap='magma'); ax1.set_title('input (noisy)')
    ax2.imshow(blurred.numpy(), cmap='magma'); ax2.set_title('ex4 5x5 box blur via patches')
    for a in (ax1, ax2): a.axis('off')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_image_patches(img: Tensor, K: int) -> Tensor:
    H, W = img.shape
    out_H, out_W = H - K + 1, W - K + 1
    sH, sW = img.stride()                              # (W, 1) for contiguous
    return img.as_strided(
        size=(out_H, out_W, K, K),
        stride=(sH, sW, sH, sW),
    )
```

**The stride pattern is *the same tuple twice*.** Outer two axes step the patch origin (`sH`, `sW`); inner two axes step inside a patch (`sH`, `sW`). They use the same underlying strides because both motions live in the same 2-D storage. This is why `as_strided` shines: a 1-line view that replaces a 4-deep nested loop.

**Why this is the im2col primitive.** A 2-D convolution is `patches @ kernel.flatten()` after this view + a `.reshape(out_H * out_W, K * K)`. Real Conv2d uses a fused kernel, but `Conv2d.forward` is morally this expression. ARENA's `conv2d_minimal` builds exactly this view; you just wrote it.

**Out-of-bounds reads are the silent killer.** `as_strided` does not check that your stride tuple stays inside `img.storage()` — if you put `K > min(H, W)` here you get a corrupt view that reads past the buffer end. Always guard with `K <= min(H, W)` in production.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()